# P3 / 01: same-slot persistence in half-hour returns (faithful regressions)

Protocol: [`001_claude_p3_intraday_periodicity_preregistration.md`](../../discussions/Intraday_Periodicity/001_claude_p3_intraday_periodicity_preregistration.md), section 4. Paper: Heston, Korajczyk & Sadka (2010).

For each 30-minute interval and lag k = 1 … 520 (40 sessions): cross-sectional slope of the interval return on the return k intervals earlier; Fama–MacBeth means with Newey–West (13) t-statistics.

**Gates** per window (one-sided, α = 0.05): **S1** the mean daily-multiple slope (k = 13, 26, …, 520) > 0; **S2** it exceeds the mean slope at non-multiple lags 14 … 520. `REPLICATES` only if both pass in **both** windows (A 2020–2025 and B 2017–2019).

In [ ]:
import json, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, p1, p2, p3, s04, s05b
from equity_trading.alpaca import ALPACA_VERSION
from equity_trading.data import vendor_dir, work_dir

SYNTHETIC = os.environ.get("P3_SYNTHETIC") == "1"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
CFG = p3.P3Config()
WINDOWS = {"A 2020-2025": list(range(2020, 2026)), "B 2017-2019": [2017, 2018, 2019]}   # 2026 sealed
WCOL = {"A 2020-2025": "#2a78d6", "B 2017-2019": "#eb6834"}
QUOTE_CACHE = vendor_dir("Alpaca") / "quote_samples" / "15Min_w3s"
CACHE = work_dir() / "results" / "p3"
CACHE.mkdir(parents=True, exist_ok=True)
if SYNTHETIC:
    CFG = replace(CFG, min_names=20, min_names_portfolio=20, n_boot=2000)
OUT = ROOT / "notebooks" / "Intraday_Periodicity" / ("01_synthetic" if SYNTHETIC else "01")
OUT.mkdir(parents=True, exist_ok=True)
print(f"synthetic={SYNTHETIC} | lags 1..{13 * CFG.max_lag_days} | windows {list(WINDOWS)}")

## 1. Data

In [ ]:
# 30-minute intervals per window: bars and quote samples are read from the caches (nothing new is downloaded)
t0 = time.time()
DATA = {}
if not SYNTHETIC:
    constituents = pd.read_csv(ROOT / "src" / "equity_trading" / "reference" / "sp500_constituents_20260913.csv")
    from equity_trading.s05 import SECTOR_ETF
    STOCKS = sorted(constituents["Symbol"])
    ETFS = sorted(set(constituents["GICS Sector"].map(SECTOR_ETF)))
for k, (name, years) in enumerate(WINDOWS.items()):
    if SYNTHETIC:
        panel, stocks = s04.synthetic_ohlc_panel([2024, 2025] if k == 0 else [2022, 2023], seed=3 + k)
        H = np.full(panel.c.shape, 2e-4, np.float32)
    else:
        if name.startswith("A"):
            panel = s04.cached_panel(STOCKS + ETFS + ["SPY"], "15Min", years)            # shared 2020-2025 cache
        else:
            panel = p2.panel_cache(STOCKS + ETFS + ["SPY"], "15Min", years, CACHE / "panel_15Min_2017_2019.npz")
        stocks = [s for s in STOCKS if s in set(panel.symbols)]
        sess = s05b.sample_sessions(years, 4, seed=42)
        missing = [d for d in sess if not (QUOTE_CACHE / f"{d}.parquet").exists()]
        assert not missing, f"quote samples missing for {missing}: run S05b / P2-04 first"
        samples = pd.concat([pd.read_parquet(QUOTE_CACHE / f"{d}.parquet") for d in sess], ignore_index=True)
        H = s05b.cost_array(panel.index, panel.day_start, panel.symbols, s05b.cost_model(samples))
    cols = np.array([panel.symbols.index(s) for s in stocks])
    d = p3.to_30m(panel, cols, H)
    if SYNTHETIC:                                    # plant periodicity so the smoke test exercises the gates
        d["R"] = p3.synthetic_intervals(len(d["days"]), len(cols), effect=1.5, seed=7 + k)
    d["stocks"] = stocks
    DATA[name] = d
    del panel, H
    print(f"{name}: {len(d['days'])} full sessions x 13 intervals, {len(stocks)} stocks, "
          f"cost median {1e4 * np.nanmedian(d['H']):.2f} bps one-way ({time.time() - t0:.0f}s)", flush=True)

## 2. Fama–MacBeth slopes by lag

In [ ]:
LAGS = np.arange(1, p3.PER_DAY * CFG.max_lag_days + 1)
MEAN, GATES, STATS = {}, {}, {}
for name, d in DATA.items():
    t0 = time.time()
    S = p3.fm_slopes(d["R"], LAGS, CFG.min_names)
    MEAN[name] = pd.Series(np.nanmean(S, 0), index=LAGS)
    Dt, Nt = p3.periodicity_series(S, LAGS)
    s1, s2 = p3.nw_mean(Dt, CFG.nw_lags), p3.nw_mean(Dt - Nt, CFG.nw_lags)
    STATS[name] = {"S1_daily_multiple_mean_slope": s1, "S2_multiple_minus_other": s2,
                   "lag1": float(MEAN[name].loc[1]), "lag13": float(MEAN[name].loc[13]), "lag26": float(MEAN[name].loc[26])}
    GATES[name] = p3.gates_01(s1, s2, CFG.alpha)
    print(f"{name}: slopes in {time.time() - t0:.0f}s")
    del S
pd.DataFrame(MEAN).to_csv(OUT / "mean_slope_by_lag.csv", float_format="%.6g")
display(pd.DataFrame({n: {"S1 mean": s["S1_daily_multiple_mean_slope"]["mean"], "S1 t": s["S1_daily_multiple_mean_slope"]["t"],
                          "S2 mean": s["S2_multiple_minus_other"]["mean"], "S2 t": s["S2_multiple_minus_other"]["t"],
                          "slope lag 1": s["lag1"], "slope lag 13": s["lag13"], "slope lag 26": s["lag26"]} for n, s in STATS.items()}).round(5))
fig, ax = plt.subplots(len(MEAN), 1, figsize=(12, 3.2 * len(MEAN)), sharex=True)
for a, (name, m) in zip(np.atleast_1d(ax), MEAN.items()):
    mult = (m.index % p3.PER_DAY == 0)
    a.bar(m.index[~mult], m[~mult], width=1.0, color="#c9c9c4")
    a.bar(m.index[mult], m[mult], width=2.5, color=WCOL[name], label="same slot, j sessions earlier")
    a.axhline(0, color="#444", lw=0.6); a.set_title(f"{name}: mean cross-sectional slope by lag (30-min intervals)"); a.legend(frameon=False)
np.atleast_1d(ax)[-1].set_xlabel("lag (intervals; 13 = one session)"); plt.tight_layout(); plt.show()

## 3. Verdict

In [ ]:
both = all(g["pass"] for g in GATES.values())
one = any(g["pass"] for g in GATES.values())
VERDICT = "REPLICATES" if both else ("PARTIAL" if one else "NO_REPLICATION")
display(pd.DataFrame(GATES)); print("verdict:", VERDICT)
summary = {"study": "P3-01", "synthetic": SYNTHETIC, "config": CFG.as_dict(), "windows": WINDOWS,
           "n_stocks": {n: len(d["stocks"]) for n, d in DATA.items()}, "n_sessions": {n: len(d["days"]) for n, d in DATA.items()},
           "stats": STATS, "gates": GATES, "verdict": VERDICT,
           "versions": {"package": __version__, "alpaca": ALPACA_VERSION, "p3": p3.P3_VERSION}}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
(OUT / "manifest.json").write_text(json.dumps({"files": sorted(p.name for p in OUT.iterdir())}, indent=2))